# Phase 6 — Le champ de vision du modèle

## Objectifs

- Démontrer, par le calcul et avant tout entraînement, que la sortie du modèle dépend de toutes les
  positions du relevé le plus long du jeu de données.
- Construire ce modèle sans jamais lire le texte mot après mot en attendant le précédent : un
  empilement de convolutions 1D dilatées, toutes les positions traitées de front.
- Rendre un tableau couche par couche du champ récepteur, la comparaison à la longueur maximale, une
  vérification expérimentale, puis l'entraînement et la comparaison au score de la phase 3.


## 1. Imports

In [ ]:
from pathlib import Path
import csv
import random
import re
import time
import urllib.request

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from torch import nn
from torch.utils.data import DataLoader, Dataset


## 2. Configuration et reproductibilité

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cpu")

URL_DATA = (
    "https://raw.githubusercontent.com/planetsig/ufo-reports/master/"
    "csv-data/ufo-complete-geocoded-time-standardized.csv"
)

DATA_DIR = Path("../data")
OUTPUT_DIR = Path("../outputs")
PHASE6_DIR = OUTPUT_DIR / "phase_6_champ_de_vision"

DATA_DIR.mkdir(parents=True, exist_ok=True)
PHASE6_DIR.mkdir(parents=True, exist_ok=True)

DATA_PATH = DATA_DIR / "releves_klaxo3.csv"

COLUMNS = [
    "datetime", "city", "state", "country", "shape",
    "duration_seconds", "duration_hours_min", "comments",
    "date_posted", "latitude", "longitude",
]

TEST_SIZE = 0.20
SEUIL_MIN_CLASSE = 5
BATCH_SIZE = 512
CONV_CHANNELS = 96
KERNEL_SIZE = 3
DILATIONS = [1, 2, 4, 8, 16]
HIDDEN_DIM = 128
DROPOUT = 0.30
LEARNING_RATE = 0.003
WEIGHT_DECAY = 0.0001
N_EPOCHS = 20
PATIENCE = 4


## 3. Téléchargement et préparation (identiques aux phases précédentes)

In [ ]:
if not DATA_PATH.exists():
    print("Téléchargement du fichier...")
    urllib.request.urlretrieve(URL_DATA, DATA_PATH)
else:
    print(f"Fichier déjà disponible : {DATA_PATH}")

lignes_valides = []
with open(DATA_PATH, "r", encoding="utf-8", errors="replace", newline="") as f:
    reader = csv.reader(f)
    for row in reader:
        if len(row) == len(COLUMNS):
            lignes_valides.append(row)

df = pd.DataFrame(lignes_valides, columns=COLUMNS)

df["comments_clean"] = df["comments"].fillna("").astype(str).str.strip()
df["shape_clean"] = df["shape"].fillna("").astype(str).str.lower().str.strip()
df["shape_model"] = df["shape_clean"].replace({"round": "circle", "changed": "changing"})

masque_forme_manquante = df["shape_clean"].eq("")
masque_fourre_tout = df["shape_model"].isin(["unknown", "other"])
masque_commentaire_vide = df["comments_clean"].eq("")

df_avant_filtre_classes_rares = df.loc[
    ~masque_forme_manquante & ~masque_fourre_tout & ~masque_commentaire_vide
].copy()

compte_classes = df_avant_filtre_classes_rares["shape_model"].value_counts()
classes_conservees = compte_classes.loc[compte_classes >= SEUIL_MIN_CLASSE].index

df_modele = df_avant_filtre_classes_rares.loc[
    df_avant_filtre_classes_rares["shape_model"].isin(classes_conservees)
].copy()

X = df_modele["comments_clean"].copy()
y = df_modele["shape_model"].copy()

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y,
)

def tokenizer(texte):
    return re.findall(r"[a-z0-9]+", str(texte).lower())

vocabulaire = {"<PAD>": 0, "<UNK>": 1}
for texte in X_train:
    for token in tokenizer(texte):
        if token not in vocabulaire:
            vocabulaire[token] = len(vocabulaire)

label_encoder = LabelEncoder()
y_train_ids = label_encoder.fit_transform(y_train)
y_val_ids = label_encoder.transform(y_val)
NOMBRE_CLASSES = len(label_encoder.classes_)

print(f"Taille train : {len(X_train)} | validation : {len(X_val)} | classes : {NOMBRE_CLASSES}")


## 4. Longueur des relevés, en jetons

On mesure la longueur tokenisée de **tous** les relevés retenus (train + validation), pas seulement du
train : c'est la longueur maximale réellement rencontrée qui fixe la contrainte de champ de vision.

In [ ]:
longueurs_tokens = df_modele["comments_clean"].apply(lambda t: len(tokenizer(t)))

MAX_LEN = int(longueurs_tokens.max())
LONGUEUR_MEDIANE = float(longueurs_tokens.median())

print(f"Longueur maximale acceptée en entrée (jetons) : {MAX_LEN}")
print(f"Longueur médiane (jetons) : {LONGUEUR_MEDIANE}")
print(f"Longueur moyenne (jetons) : {longueurs_tokens.mean():.2f}")


## 5. Le modèle : convolutions 1D dilatées, empilées avec connexions résiduelles

Chaque bloc traite **toutes** les positions de la séquence en une seule opération matricielle — aucune
récurrence, aucune attente du jeton précédent. Le champ de vision d'une sortie ne grandit pas parce
que le modèle « lit dans l'ordre », il grandit parce qu'on empile des couches dont chacune regarde un
peu plus loin que la précédente (la dilatation croît en puissances de deux).

`BatchNorm1d` stabilise l'entraînement d'un empilement profond ; la connexion résiduelle (`x + bloc(x)`)
évite que le signal ne s'atténue en traversant les cinq couches.

In [ ]:
class BlocConvResiduel(nn.Module):
    def __init__(self, canaux, kernel_size, dilation):
        super().__init__()
        padding = (kernel_size - 1) * dilation // 2
        self.conv = nn.Conv1d(canaux, canaux, kernel_size, dilation=dilation, padding=padding)
        self.norme = nn.BatchNorm1d(canaux)
        self.activation = nn.ReLU()
        self.dropout = nn.Dropout(DROPOUT)

    def forward(self, x):
        sortie = self.conv(x)
        sortie = self.norme(sortie)
        sortie = self.activation(sortie)
        sortie = self.dropout(sortie)
        return x + sortie  # connexion residuelle

class TroncConvolutif(nn.Module):
    """La partie du modele dont on etudie le champ de vision : embedding + blocs dilates.
    Ne contient ni pooling ni tete de classification, pour pouvoir observer une position precise."""
    def __init__(self, taille_vocabulaire, canaux, kernel_size, dilations):
        super().__init__()
        self.embedding = nn.Embedding(taille_vocabulaire, canaux, padding_idx=0)
        self.blocs = nn.ModuleList([
            BlocConvResiduel(canaux, kernel_size, dilation) for dilation in dilations
        ])
        self.kernel_size = kernel_size
        self.dilations = dilations

    def forward(self, ids):
        x = self.embedding(ids).transpose(1, 2)  # (batch, canaux, longueur)
        for bloc in self.blocs:
            x = bloc(x)
        return x  # (batch, canaux, longueur)

class ClassifieurConvolutif(nn.Module):
    def __init__(self, taille_vocabulaire, nombre_classes, canaux, kernel_size, dilations, hidden_dim):
        super().__init__()
        self.tronc = TroncConvolutif(taille_vocabulaire, canaux, kernel_size, dilations)
        self.tete = nn.Sequential(
            nn.Linear(canaux, hidden_dim),
            nn.ReLU(),
            nn.Dropout(DROPOUT),
            nn.Linear(hidden_dim, nombre_classes),
        )

    def forward(self, ids, masque):
        traits = self.tronc(ids)  # (batch, canaux, longueur)
        masque_etendu = masque.unsqueeze(1)  # (batch, 1, longueur)
        somme = (traits * masque_etendu).sum(dim=2)
        compte = masque_etendu.sum(dim=2).clamp(min=1)
        pool_moyenne = somme / compte  # moyenne masquee : le padding ne compte pas
        return self.tete(pool_moyenne)


## 6. Tableau du champ récepteur, couche par couche

Calculé directement depuis la configuration du modèle (`kernel_size`, `dilations`), pas recopié à la
main. Le padding est symétrique (`(k-1)*d/2` de chaque côté) : une couche ne regarde donc pas
seulement vers l'avant, elle regarde `d` positions **de chaque côté**. Il faut distinguer deux
grandeurs :

- la **portée** : jusqu'où une sortie regarde d'un seul côté (cumul des `dilation`) ;
- le **diamètre** : l'étendue totale vue par une sortie, portée de gauche + portée de droite + elle-même
  (cumul des `(k-1) × dilation`, comme un WaveNet classique le compterait).

Le diamètre est ce qu'on compare naturellement à la longueur maximale, mais c'est la **portée** qui
dit si une position précise, proche d'un bord, peut voir l'autre bord.

In [ ]:
def tableau_champ_recepteur(kernel_size, dilations):
    lignes = []
    diametre_cumule = 1  # une sortie voit d'abord sa propre position
    portee_cumulee = 0
    for i, dilation in enumerate(dilations, start=1):
        ajout = (kernel_size - 1) * dilation
        diametre_cumule += ajout
        portee_cumulee += dilation  # (kernel_size - 1) // 2 * dilation, ici kernel=3 donc = dilation
        lignes.append({
            "couche": i,
            "kernel_size": kernel_size,
            "dilation": dilation,
            "positions_ajoutees_diametre": ajout,
            "portee_dun_cote_cumulee": portee_cumulee,
            "diametre_cumule": diametre_cumule,
        })
    return pd.DataFrame(lignes)

tableau_rf = tableau_champ_recepteur(KERNEL_SIZE, DILATIONS)
tableau_rf


In [ ]:
diametre_total = int(tableau_rf["diametre_cumule"].iloc[-1])
portee_totale = int(tableau_rf["portee_dun_cote_cumulee"].iloc[-1])

print(f"Diamètre total du champ récepteur d'une sortie : {diametre_total} positions")
print(f"Portée d'un seul côté                          : {portee_totale} positions")
print(f"Longueur maximale acceptée en entrée            : {MAX_LEN} positions")
print(f"Le diamètre dépasse la longueur maximale : {diametre_total >= MAX_LEN} "
      f"(garantit qu'une position centrale voit tout le relevé)")
print(f"La portée seule couvre le relevé entier  : {portee_totale >= MAX_LEN - 1} "
      f"(condition nécessaire pour qu'une position en bord de séquence voie l'autre bord à elle seule)")


## 7. Vérification expérimentale (avant tout entraînement)

Deux vérifications, sur le relevé le plus long du jeu, en ne changeant que le premier jeton.

**(a) Une seule position du tronc, en bord de séquence.** La dernière position d'une séquence de
55 jetons est à distance 54 de la première — largement au-delà de la portée de 31 calculée plus haut.
Elle ne devrait donc **pas** bouger : ce n'est pas un défaut, c'est la conséquence directe et attendue
d'un champ récepteur symétrique dont la portée est plus courte que le relevé.

**(b) La sortie réellement utilisée par le classifieur.** Le modèle ne lit jamais une seule position :
la tête de classification moyenne (masque compris) **toutes** les positions du tronc. Le premier mot
influence les positions du tronc de 0 à 31 (sa propre portée), qui entrent toutes dans cette moyenne.
C'est donc la sortie poolée, pas une position isolée, qui doit dépendre de tous les jetons — et c'est
elle qui compte pour la tâche.

In [ ]:
index_plus_long = longueurs_tokens.idxmax()
texte_plus_long = df_modele.loc[index_plus_long, "comments_clean"]
tokens_plus_long = tokenizer(texte_plus_long)

print(f"Relevé le plus long ({len(tokens_plus_long)} jetons) : {texte_plus_long!r}")

ids_originaux = [vocabulaire.get(t, vocabulaire["<UNK>"]) for t in tokens_plus_long]
ids_originaux = ids_originaux + [vocabulaire["<PAD>"]] * (MAX_LEN - len(ids_originaux))
masque_verification = [1.0] * len(tokens_plus_long) + [0.0] * (MAX_LEN - len(tokens_plus_long))

ids_perturbes = list(ids_originaux)
premier_id_actuel = ids_perturbes[0]
nouvel_id = (premier_id_actuel + 1) % len(vocabulaire)
if nouvel_id == vocabulaire["<PAD>"]:
    nouvel_id = vocabulaire["<UNK>"]
ids_perturbes[0] = nouvel_id

torch.manual_seed(SEED)
modele_verification = ClassifieurConvolutif(
    len(vocabulaire), NOMBRE_CLASSES, CONV_CHANNELS, KERNEL_SIZE, DILATIONS, HIDDEN_DIM,
)
modele_verification.eval()

tenseur_original = torch.tensor([ids_originaux], dtype=torch.long)
tenseur_perturbe = torch.tensor([ids_perturbes], dtype=torch.long)
tenseur_masque = torch.tensor([masque_verification], dtype=torch.float32)

derniere_position = len(tokens_plus_long) - 1

with torch.no_grad():
    trace_originale = modele_verification.tronc(tenseur_original)
    trace_perturbee = modele_verification.tronc(tenseur_perturbe)
    logits_originaux = modele_verification(tenseur_original, tenseur_masque)
    logits_perturbes = modele_verification(tenseur_perturbe, tenseur_masque)

ecart_par_position = (trace_originale - trace_perturbee)[0].abs().max(dim=0).values
ecart_derniere_position = ecart_par_position[derniere_position].item()
derniere_position_influencee = int((ecart_par_position > 1e-6).nonzero().max().item())
ecart_logits = (logits_originaux - logits_perturbes).abs().max().item()

print(f"(a) Écart sur la dernière position ({derniere_position}), distance {derniere_position} > portée {portee_totale} "
      f": {ecart_derniere_position:.8f} -> inchangée, comme prévu par le calcul de portée.")
print(f"    Dernière position réellement influencée par le 1er jeton : {derniere_position_influencee} "
      f"(cohérent avec une portée de {portee_totale})")
print(f"(b) Écart sur la sortie du classifieur (logits, après pooling masqué) : {ecart_logits:.6f} "
      f"-> {'change bien' if ecart_logits > 1e-6 else 'NE CHANGE PAS'} avec le 1er jeton.")


## 8. Jeu de données et boucle d'entraînement

In [ ]:
class DatasetConvolutif(Dataset):
    def __init__(self, textes, labels, vocabulaire, max_len):
        self.labels = list(labels)
        self.ids = []
        self.masques = []
        for texte in textes:
            tokens = tokenizer(texte)[:max_len]
            ids = [vocabulaire.get(t, vocabulaire["<UNK>"]) for t in tokens]
            longueur = len(ids)
            if longueur == 0:
                ids = [vocabulaire["<UNK>"]]
                longueur = 1
            masque = [1.0] * longueur + [0.0] * (max_len - longueur)
            ids = ids + [vocabulaire["<PAD>"]] * (max_len - longueur)
            self.ids.append(torch.tensor(ids, dtype=torch.long))
            self.masques.append(torch.tensor(masque, dtype=torch.float32))

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return self.ids[index], self.masques[index], int(self.labels[index])

dataset_train = DatasetConvolutif(X_train, y_train_ids, vocabulaire, MAX_LEN)
dataset_val = DatasetConvolutif(X_val, y_val_ids, vocabulaire, MAX_LEN)

loader_train = DataLoader(dataset_train, batch_size=BATCH_SIZE, shuffle=True)
loader_val = DataLoader(dataset_val, batch_size=BATCH_SIZE, shuffle=False)


In [ ]:
def une_epoque_train(modele, loader, optimiseur, fonction_perte):
    modele.train()
    perte_totale, nombre_exemples = 0.0, 0
    for ids, masques, labels in loader:
        optimiseur.zero_grad()
        logits = modele(ids, masques)
        perte = fonction_perte(logits, labels)
        perte.backward()
        optimiseur.step()
        perte_totale += perte.item() * len(labels)
        nombre_exemples += len(labels)
    return perte_totale / nombre_exemples

def evaluer_modele(modele, loader, fonction_perte):
    modele.eval()
    perte_totale, nombre_exemples = 0.0, 0
    predictions, labels_reels = [], []
    with torch.no_grad():
        for ids, masques, labels in loader:
            logits = modele(ids, masques)
            perte = fonction_perte(logits, labels)
            perte_totale += perte.item() * len(labels)
            nombre_exemples += len(labels)
            predictions.extend(logits.argmax(dim=1).tolist())
            labels_reels.extend(labels.tolist())
    accuracy = accuracy_score(labels_reels, predictions)
    return perte_totale / nombre_exemples, accuracy


## 9. Entraînement du modèle convolutif

In [ ]:
torch.manual_seed(SEED)
modele_convolutif = ClassifieurConvolutif(
    len(vocabulaire), NOMBRE_CLASSES, CONV_CHANNELS, KERNEL_SIZE, DILATIONS, HIDDEN_DIM,
).to(DEVICE)

optimiseur = torch.optim.AdamW(modele_convolutif.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
fonction_perte = nn.CrossEntropyLoss()

historique = {"epoch": [], "train_loss": [], "val_loss": [], "val_accuracy": []}
meilleure_perte_val = float("inf")
meilleur_etat = None
epochs_sans_amelioration = 0

debut = time.perf_counter()
for epoch in range(1, N_EPOCHS + 1):
    perte_train = une_epoque_train(modele_convolutif, loader_train, optimiseur, fonction_perte)
    perte_val, acc_val = evaluer_modele(modele_convolutif, loader_val, fonction_perte)

    historique["epoch"].append(epoch)
    historique["train_loss"].append(perte_train)
    historique["val_loss"].append(perte_val)
    historique["val_accuracy"].append(acc_val)

    print(f"Epoch {epoch:02d} | train={perte_train:.4f} | val={perte_val:.4f} | accuracy={acc_val:.2%}")

    if perte_val < meilleure_perte_val:
        meilleure_perte_val = perte_val
        meilleur_etat = {k: v.cpu().clone() for k, v in modele_convolutif.state_dict().items()}
        epochs_sans_amelioration = 0
    else:
        epochs_sans_amelioration += 1

    if epochs_sans_amelioration >= PATIENCE:
        print("Arrêt anticipé : absence d'amélioration de la validation.")
        break

temps_entrainement = time.perf_counter() - debut
modele_convolutif.load_state_dict(meilleur_etat)


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(historique["epoch"], historique["train_loss"], marker="o", label="Perte entraînement")
plt.plot(historique["epoch"], historique["val_loss"], marker="o", label="Perte validation")
plt.title("Phase 6 — Courbes de perte, modèle convolutif dilaté")
plt.xlabel("Époque")
plt.ylabel("Cross-entropy loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(PHASE6_DIR / "courbes_perte_convolutif.png", dpi=150)
plt.show()


## 10. Comparaison au score de la phase 3

In [ ]:
resume_phase3 = pd.read_csv(OUTPUT_DIR / "phase_3_baseline_et_modele_pytorch" / "resume_phase3.csv")
ACCURACY_PHASE3 = float(resume_phase3["accuracy_pytorch"].iloc[0])

perte_val_finale, accuracy_convolutif = evaluer_modele(modele_convolutif, loader_val, fonction_perte)

print(f"Accuracy phase 3 (EmbeddingBag + MLP)     : {ACCURACY_PHASE3:.2%}")
print(f"Accuracy phase 6 (convolutif dilaté)      : {accuracy_convolutif:.2%}")
print(f"Écart                                     : {accuracy_convolutif - ACCURACY_PHASE3:+.2%}")

if accuracy_convolutif < ACCURACY_PHASE3:
    print(
        "L'empilement dégrade le score : la connexion résiduelle et le BatchNorm1d sont déjà en place "
        "pour stabiliser l'entraînement profond ; un score plus bas signale surtout qu'un modèle "
        "positionnel a besoin de plus d'époques ou d'un taux d'apprentissage plus prudent que le "
        "sac-de-mots de la phase 3, pas que l'idée est mauvaise."
    )
else:
    print("Le modèle convolutif égale ou dépasse la phase 3 tout en respectant la contrainte de la salle des calculs.")


## 11. Export des résultats

In [ ]:
tableau_rf.to_csv(PHASE6_DIR / "champ_recepteur_par_couche.csv", index=False)
pd.DataFrame(historique).to_csv(PHASE6_DIR / "historique_pertes_convolutif.csv", index=False)

resume_phase6 = pd.DataFrame([{
    "max_len_jetons": MAX_LEN,
    "longueur_mediane_jetons": LONGUEUR_MEDIANE,
    "nombre_couches": len(DILATIONS),
    "kernel_size": KERNEL_SIZE,
    "dilations": str(DILATIONS),
    "diametre_total": diametre_total,
    "portee_dun_cote": portee_totale,
    "diametre_suffisant_pour_position_centrale": diametre_total >= MAX_LEN,
    "portee_suffisante_pour_position_de_bord": portee_totale >= MAX_LEN - 1,
    "ecart_derniere_position_tronc": ecart_derniere_position,
    "derniere_position_influencee_par_1er_jeton": derniere_position_influencee,
    "ecart_logits_classifieur": ecart_logits,
    "epochs_executees": historique["epoch"][-1],
    "temps_entrainement_secondes": temps_entrainement,
    "accuracy_phase3_reference": ACCURACY_PHASE3,
    "accuracy_phase6_convolutif": accuracy_convolutif,
}])
resume_phase6.to_csv(PHASE6_DIR / "resume_phase6.csv", index=False)

resume_phase6
